In [ ]:
! pip install faiss-cpu

In [ ]:
!conda install -c pytorch -c nvidia faiss-gpu=1.8.0 pytorch-cuda=11.8

/bin/bash: line 1: conda: command not found


In [ ]:
!pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 785.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 85.3 MB/s eta 0:00:00


In [ ]:
! pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Fo

In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install numpy

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

documents = [
    "The cat sits on the mat.",
    "A dog runs in the park.",
    "I love eating pizza with pineapple.",
    "Machine learning is fascinating.",
    "Vector databases enable semantic search.",
    "The weather today is sunny and warm.",
    "My favorite color is blue."
]

# Generate embeddings (as numpy arrays)
embeddings = model.encode(documents)
print(f"Embedding shape: {embeddings.shape}")  # (7, 384)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding shape: (7, 384)


In [ ]:
import faiss

# Build index
dimension = embeddings.shape[1]
index_flat = faiss.IndexFlatL2(dimension)   # L2 distance
print(f"Index is trained? {index_flat.is_trained}")  # Flat indexes don't need training

# Add vectors
index_flat.add(embeddings)
print(f"Number of vectors: {index_flat.ntotal}")

# Search
query = model.encode(["A cat playing outside"])[0]
k = 3
distances, indices = index_flat.search(query.reshape(1, -1), k)
print("Flat L2 results:")
print("Indices:", indices)
print("Distances:", distances)

Index is trained? True
Number of vectors: 7
Flat L2 results:
Indices: [[0 1 5]]
Distances: [[0.91089636 1.4215158  1.691432  ]]


In [ ]:
import faiss
nlist = 2  # number of clusters (should be sqrt(N) roughly)
quantizer = faiss.IndexFlatL2(dimension)  # the quantizer is another index
index_ivf = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_L2)

# IVF indexes must be trained
assert not index_ivf.is_trained
index_ivf.train(embeddings)
assert index_ivf.is_trained

# Add vectors
index_ivf.add(embeddings)

# Set number of probes (how many clusters to examine)
index_ivf.nprobe = 2

# Search
distances_ivf, indices_ivf = index_ivf.search(query.reshape(1, -1), k)
print("IVF results:")
print("Indices:", indices_ivf)
print("Distances:", distances_ivf)

IVF results:
Indices: [[0 1 5]]
Distances: [[0.91089636 1.4215158  1.691432  ]]


In [ ]:
# HNSW index
index_hnsw = faiss.IndexHNSWFlat(dimension, 32)  # 32 is the number of neighbors per node
index_hnsw.hnsw.efConstruction = 40  # higher = more accurate construction, default is 40
index_hnsw.add(embeddings)

# Set search depth
index_hnsw.hnsw.efSearch = 16  # higher = more accurate but slower search

distances_hnsw, indices_hnsw = index_hnsw.search(query.reshape(1, -1), k)
print("HNSW results:")
print("Indices:", indices_hnsw)
print("Distances:", distances_hnsw)

HNSW results:
Indices: [[0 1 5]]
Distances: [[0.91089636 1.4215158  1.691432  ]]


In [ ]:
import time

# Create a larger random dataset
d = 384
nb = 10000  # database size
nq = 100    # number of queries
np.random.seed(1234)
xb = np.random.random((nb, d)).astype('float32')
xq = np.random.random((nq, d)).astype('float32')

# Flat index
index_flat = faiss.IndexFlatL2(d)
index_flat.add(xb)
start = time.time()
index_flat.search(xq, 10)
flat_time = time.time() - start

# IVF index (nlist=100, nprobe=5)
quantizer = faiss.IndexFlatL2(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, 100)
index_ivf.train(xb)
index_ivf.add(xb)
index_ivf.nprobe = 5
start = time.time()
index_ivf.search(xq, 10)
ivf_time = time.time() - start

# HNSW index (neighbors=32, efSearch=16)
index_hnsw = faiss.IndexHNSWFlat(d, 32)
index_hnsw.add(xb)
index_hnsw.hnsw.efSearch = 16
start = time.time()
index_hnsw.search(xq, 10)
hnsw_time = time.time() - start

print(f"Flat time: {flat_time*1000:.2f} ms")
print(f"IVF time:  {ivf_time*1000:.2f} ms")
print(f"HNSW time: {hnsw_time*1000:.2f} ms")

Flat time: 46.20 ms
IVF time:  15.92 ms
HNSW time: 27.82 ms


In [ ]:
# Chroma DB

In [ ]:
import chromadb
from chromadb.config import Settings

# Create a persistent client (data saved to disk)
client = chromadb.PersistentClient(path="./chroma_data")

# Or use an in‑memory client for testing
# client = chromadb.EphemeralClient()

# Create a collection
collection = client.create_collection(
    name="my_docs",
    metadata={"hnsw:space": "l2"}  # default distance is "l2", can also be "cosine" or "ip"
)

# Add documents (Chroma can compute embeddings automatically)
collection.add(
    documents=documents,
    metadatas=[{"source": f"doc{i}"} for i in range(len(documents))],
    ids=[f"id{i}" for i in range(len(documents))]
)
# By default, Chroma uses all‑MiniLM‑L6‑v2 to compute embeddings.

# Query
results = collection.query(
    query_texts=["A cat in the house"],
    n_results=3
)
print("Query results:", results)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 41.6MiB/s]


Query results: {'ids': [['id0', 'id1', 'id4']], 'embeddings': None, 'documents': [['The cat sits on the mat.', 'A dog runs in the park.', 'Vector databases enable semantic search.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'source': 'doc0'}, {'source': 'doc1'}, {'source': 'doc4'}]], 'distances': [[0.8662965297698975, 1.734896183013916, 1.9037878513336182]]}


In [ ]:
collection_cosine = client.create_collection(
    name="cosine_docs",
    metadata={"hnsw:space": "cosine"}
)

In [ ]:
# Add documents with metadata
collection.add(
    documents=["I love dogs", "I love cats", "Dogs are great", "Cats are independent"],
    metadatas=[
        {"animal": "dog", "sentiment": "positive"},
        {"animal": "cat", "sentiment": "positive"},
        {"animal": "dog", "sentiment": "positive"},
        {"animal": "cat", "sentiment": "neutral"}
    ],
    ids=["id10", "id11", "id12", "id13"]
)

# Query with filter: only documents about dogs
results = collection.query(
    query_texts=["great pets"],
    n_results=2,
    where={"animal": "dog"}   # metadata filter
)
print("Filtered results:", results)

Filtered results: {'ids': [['id12', 'id10']], 'embeddings': None, 'documents': [['Dogs are great', 'I love dogs']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'animal': 'dog', 'sentiment': 'positive'}, {'sentiment': 'positive', 'animal': 'dog'}]], 'distances': [[0.4045846462249756, 0.7192965149879456]]}
